# 📦 ACE-Net Master Dataset Archiver (Zero-Compression Turbo Zip)
### Lumilikha ng `baseline_features_all.zip` at nag-a-upload sa Google Drive

### 💡 Bakit Turbo Fast ito:
1. **Store-Only (`zip -0`):** Hindi na kailangan ng CPU compression dahil naka-compress na ang mga `.jpg` images at `.npy` arrays. Diretsong data streaming lang sa pinakamataas na bilis!
2. **Live Progress Tracking:** Makikita mo ang bawat folder (`VAL`, `TEST`, `TRAIN`) habang ini-i-zip nang may status at elapsed time.
3. **1-Click Upload:** Pagkatapos ma-archive sa local SSD, awtomatiko itong ia-upload sa Google Drive mo bilang nag-iisang `baseline_features_all.zip`.

> **Resulta para sa Training:** Pagkatapos nito, sa `TRAIN_BASELINE_ACENET.ipynb`, **20 seconds na lang ang unzip** papuntang local SSD at haharurot na ang training sa 0.2s bawat batch!

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
print('✅ Google Drive mounted successfully!')

## Step 2: Turbo-Zip All Folders (VAL, TEST, TRAIN) & Upload to Google Drive

In [ ]:
import os, time, shutil
from pathlib import Path

SOURCE_ROOT = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/Baseline preprocessed')
DRIVE_DEST_DIR = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training')
FINAL_DRIVE_ZIP = DRIVE_DEST_DIR / 'baseline_features_all.zip'
LOCAL_ZIP = Path('/content/baseline_features_all.zip')

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"❌ Hindi mahanap ang path: {SOURCE_ROOT}")

subfolders = [p.name for p in SOURCE_ROOT.iterdir() if p.is_dir()]
# Order: VAL, TEST, then TRAIN for fast visual confirmation
order = ['VAL', 'TEST', 'TRAIN']
sorted_folders = [f for f in order if f in subfolders] + [f for f in subfolders if f not in order]

print('=' * 80)
print('📁 Detected Preprocessed Folders:', sorted_folders)
print(f'📍 Output Target on Drive       : {FINAL_DRIVE_ZIP}')
print('=' * 80)

start_total = time.time()

# Archive each folder progressively with live status
for idx, folder_name in enumerate(sorted_folders, 1):
    f_start = time.time()
    print(f'\n🚀 [{idx}/{len(sorted_folders)}] Archiving folder "{folder_name}" to local master zip...')
    
    # zip -0 (store only, NO CPU compression overhead)
    # -u updates the existing zip archive incrementally
    !cd "{SOURCE_ROOT}" && zip -0 -q -u -r "{LOCAL_ZIP}" "{folder_name}"
    
    f_elapsed = time.time() - f_start
    cur_size_gb = LOCAL_ZIP.stat().st_size / (1024**3) if LOCAL_ZIP.exists() else 0
    print(f'   ✅ Completed "{folder_name}" in {f_elapsed/60:.2f} mins! (Current Zip Size: {cur_size_gb:.2f} GB)')

total_zip_time = time.time() - start_total
final_local_size_gb = LOCAL_ZIP.stat().st_size / (1024**3)
print('\n' + '=' * 80)
print(f'🎉 Local Master Zip Complete! Size: {final_local_size_gb:.2f} GB in {total_zip_time/60:.2f} mins')
print('=' * 80)

# Step 2: Fast Streaming Single File Copy to Google Drive
print(f'\n🚀 Uploading single archive ({final_local_size_gb:.2f} GB) to Google Drive...')
upload_start = time.time()
shutil.copy2(str(LOCAL_ZIP), str(FINAL_DRIVE_ZIP))
upload_time = time.time() - upload_start
print(f'✅ Google Drive Upload Complete in {upload_time:.1f}s!')

# Clean up local temporary file
LOCAL_ZIP.unlink()

total_elapsed = time.time() - start_total
print('\n' + '=' * 80)
print('       🏆 MASTER ARCHIVE IS READY ON GOOGLE DRIVE! 🏆')
print('=' * 80)
print(f'📍 Drive Location    : {FINAL_DRIVE_ZIP}')
print(f'📦 Total Archive Size: {FINAL_DRIVE_ZIP.stat().st_size / (1024**3):.2f} GB')
print(f'⏱️ Total Time Taken  : {total_elapsed/60:.2f} mins')
print('=' * 80)
print('\n👉 Tapos na! Pwede mo nang buksan ang TRAIN_BASELINE_ACENET.ipynb.')
print('   20 seconds na lang ang pag-unzip doon at magsisimula na agad ang training!')